# RAG implementation

Implementing the core RAG pipeline without using any high-level RAG
framework. Everything: chunking, embedding, vector search, and generation must
be wired by you.

Step 1: Reading the document file in Python.

In [15]:
f1 = open("Document.txt", "r")

In [16]:
f1.read()

'Leadership and Management Importance in Organisations\n\n\nIn today’s increasingly complex and dynamic environments, leadership is regarded as an important part of organisational success. Management training and development is considered a vital facet of the organisational community and environment. Various studies show that companies that align their management development and strategic goals are successful in competitive business environments.\nHowever, can these organisations remain competitive without the interventions of trained managers? Companies need effective managers who hold relevant practical, interpersonal, and conceptual skills to lead organisations in complex business situations. In a fast paced world, effective organisational management is necessary for the development of a positive workplace that helps employees to realise optimal performance.\nIn an ideal world, all managers are expected to be leaders. Nevertheless, not all appointed managers possess good leadership 

Part A: Chunking
Implement fixed-size chunking with overlap.

In [17]:
def chunk_text(text: str, chunk_size: int = 200, overlap:int = 40) -> list[str]:
 """
 Split `text` into chunks of ~chunk_size words,
 with `overlap` words shared between consecutive chunks.
 Returns a list of chunk strings.
 """
 chunks = []
 words = text.split()
 for i in range(0, len(words), chunk_size - overlap):
     chunk = " ".join(words[i:i + chunk_size])
     chunks.append(chunk)
 return chunks

In [18]:
f1.seek(0)
text_content = f1.read()

chunks = chunk_text(text_content, chunk_size=200, overlap=40)
print(f"Total chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1}:\n{chunk}\n")

Total chunks: 26

Chunk 1:
Leadership and Management Importance in Organisations In today’s increasingly complex and dynamic environments, leadership is regarded as an important part of organisational success. Management training and development is considered a vital facet of the organisational community and environment. Various studies show that companies that align their management development and strategic goals are successful in competitive business environments. However, can these organisations remain competitive without the interventions of trained managers? Companies need effective managers who hold relevant practical, interpersonal, and conceptual skills to lead organisations in complex business situations. In a fast paced world, effective organisational management is necessary for the development of a positive workplace that helps employees to realise optimal performance. In an ideal world, all managers are expected to be leaders. Nevertheless, not all appointed managers posse

**Part B: Embedding & Vector Store**

Use sentence-transformers (model: all-MiniLM-L6-v2) to embed your chunks.
Store embeddings in a numpy array (no vector DB required here, just numpy).
Implement cosine similarity manually (define a function named cosine_similarity
which takes a and b as parameters)
Formula: cos(θ) = A·B / ‖A‖‖B‖). Do not use sklearn for this part.
Then: run 3 different queries against your document and print the top-3 retrieved
chunks for each. Include the cosine scores alongside.

In [19]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the model explicitly
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. Embed all chunks -> shape (num_chunks, embedding_dim)
# We add convert_to_numpy=True to ensure it yields a clean numpy array using PyTorch under the hood
chunk_embeddings = embedding_model.encode(chunks, convert_to_numpy=True)

# 2. Manual Cosine Similarity function (Formula: cos(θ) = A·B / ‖A‖‖B‖)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Computes the cosine similarity between two 1D numpy arrays."""
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    # Simple guard against division by zero
    if norm_a == 0 or norm_b == 0:
        return 0.0

    return float(dot_product / (norm_a * norm_b))

# 3. Retrieve implementation
def retrieve(query: str, top_k: int = 3) -> list[tuple[str, float]]:
    """Return the top_k most similar chunks to `query` as (chunk_text, score) tuples."""
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)

    scored_chunks = []
    for i, chunk_emb in enumerate(chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_emb)
        scored_chunks.append((chunks[i], score))

    # Sort descending based on the score
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    return scored_chunks[:top_k]

# --- Quick Test Verification ---
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print("Running a quick retrieval check...")
results = retrieve("Why is emotional intelligence important?", top_k=1)
if results:
    print(f"Top Match Score: {results[0][1]:.4f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (26, 384)
Running a quick retrieval check...
Top Match Score: 0.6658


**Part C: Generation**

Plug your retriever into an LLM call. Use the google/flan-t5-base via
HuggingFace.
Firstly, install the package by running this command in the terminal.

In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the tokenizer and model directly (bypassing the pipeline wrapper entirely)
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def rag_answer(query: str) -> str:
    # retrieve() returns a list of tuples: (chunk_text, score)
    results = retrieve(query, top_k=3)

    # Join the retrieved chunks into a single context block
    context = "\n\n".join(chunk for chunk, _ in results)

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not in the context, say 'I don't know'.

Context:
{context}

Question: {query}
Answer:"""

    # 2. Tokenize the input string into PyTorch tensors
    inputs = tokenizer(prompt, return_tensors="pt")

    # 3. Generate the response tokens
    outputs = model.generate(**inputs, max_new_tokens=150)

    # 4. Decode the raw tokens back into human-readable text
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Run 2 queries: one answerable from your document and one deliberately outside
the document's scope. Show that your pipeline returns "I don't know" (or
equivalent) for the out-of-scope query. This is the hallucination guard.


In [25]:
#Run 2 queries: one answerable from your document and one deliberately outside the document's scope. Show that your pipeline returns "I don't know" (or equivalent) for the out-of-scope query. This is the hallucination guard.
# Run the final evaluation
Queries = [
    "What is the main topic of the document?",
    "What is the airspeed of a laden European swallow?"
]

for query in Queries:
    answer = rag_answer(query)
    print(f"Q: {query}\nA: {answer}\n")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (819 > 512). Running this sequence through the model will result in indexing errors


Q: What is the main topic of the document?
A: Leadership

Q: What is the airspeed of a laden European swallow?
A: I don't know

